# Week 7 Coding Practice: Sentiment Analysis

This notebook compares two approaches to IMDb sentiment classification:
- a lexicon-based / rule-based method using VADER
- a supervised method using TF-IDF + Logistic Regression

We will focus on how the methods behave on real review text and why sentiment is often harder than it looks.

**Goal:** interpret model behavior before treating any prediction as automatically correct.

## Open in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module7/week07_sentiment_practice_solution.ipynb)

GitHub notebook: [module7/week07_sentiment_practice_solution.ipynb](https://github.com/obscrivn/mynewbook/blob/master/module7/week07_sentiment_practice_solution.ipynb)

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import movie_reviews
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

nltk.download('movie_reviews', quiet=True)
nltk.download('vader_lexicon', quiet=True)

rows = []
for fileid in movie_reviews.fileids():
    label = 'positive' if fileid.startswith('pos/') else 'negative'
    rows.append({'review': movie_reviews.raw(fileid), 'label': label})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Loaded {len(df)} movie reviews from the NLTK corpus.')
df.head()


Loaded 2000 movie reviews from the NLTK corpus.


,review,label
0,the verdict : spine-chilling drama from horror...,positive
1,""" the 44 caliber killer has struck again . "" ...",negative
2,in the company of men made a splash at the sun...,positive
3,"in the year 2029 , captain leo davidson ( mark...",negative
4,[note that followups are directed to rec . art...,positive


## Part 1: Explore the sentiment task

Before fitting any model, inspect a few positive and negative reviews. Which reviews seem easy to classify, and which look more ambiguous or difficult?

In [2]:
pd.set_option('display.max_colwidth', 180)
positive_examples = df[df['label'] == 'positive'].head(3)
negative_examples = df[df['label'] == 'negative'].head(3)

example_df = pd.concat([positive_examples, negative_examples], ignore_index=True)
example_df[['label', 'review']]


,label,review
0,positive,"the verdict : spine-chilling drama from horror maestro stephen king , featuring an outstanding , oscar-winning performance from kathy bates . \ngeez , french and saunders had a..."
1,positive,"in the company of men made a splash at the sundance film festival because , in a year plagued by brothers mcmullen-style , earnestly shallow gen-x angst pictures , it seemed to..."
2,positive,"[note that followups are directed to rec . arts . movies . current-films and rec . arts . movies . startrek . current \nonly , not to rec . arts . sf . movies . \n-moderator] \..."
3,negative,""" the 44 caliber killer has struck again . "" \nstarring john leguizamo , mira sorvino , adrian brody , jennifer esposito , michael rispoli , bebe neuwirth . \nrated r . \nsumm..."
4,negative,"in the year 2029 , captain leo davidson ( mark wahlberg , "" boogie nights "" ) is training 'his' chimp pericles to pilot a pod from the usaf oberon space station . \nwhen an ele..."
5,negative,"synopsis : a maniac , crazed by virulent microphage , slaughters more than twenty people , including a street gang and heavily-armed troops , with a small knife . \neven with a..."


### Prediction check

Before running a sentiment model, which review do you expect to be easiest to classify? Which one might be hardest, and why?

**Expected answer:** the easiest reviews are usually the most clearly positive or negative, with strong sentiment words and little mixed emotion. Hard cases often include negation, sarcasm, or a mix of praise and criticism.

## Part 2: Lexicon-based / rule-based sentiment with VADER

VADER uses a sentiment lexicon plus rules for punctuation, capitalization, and negation. It does not learn a model from labels; instead, it scores words and phrases based on their prior sentiment.

This is a good example of a lexicon-based method that is lightweight and easy to inspect.


In [3]:
analyzer = SentimentIntensityAnalyzer()

def vader_scores(review):
    scores = analyzer.polarity_scores(review)
    return {
        'neg': scores['neg'],
        'neu': scores['neu'],
        'pos': scores['pos'],
        'compound': scores['compound'],
    }

vader_df = df.copy()
vader_df[['neg', 'neu', 'pos', 'compound']] = pd.DataFrame(vader_df['review'].apply(vader_scores).tolist())

vader_df['vader_label'] = vader_df['compound'].apply(
    lambda score: 'positive' if score >= 0.05 else 'negative' if score <= -0.05 else ('positive' if score >= 0 else 'negative')
)

vader_df[['review', 'label', 'neg', 'neu', 'pos', 'compound', 'vader_label']].head(5)


,review,label,neg,neu,pos,compound,vader_label
0,"the verdict : spine-chilling drama from horror maestro stephen king , featuring an outstanding , oscar-winning performance from kathy bates . \ngeez , french and saunders had a...",positive,0.150,0.702,0.147,-0.5966,negative
1,""" the 44 caliber killer has struck again . "" \nstarring john leguizamo , mira sorvino , adrian brody , jennifer esposito , michael rispoli , bebe neuwirth . \nrated r . \nsumm...",negative,0.144,0.750,0.107,-0.9878,negative
2,"in the company of men made a splash at the sundance film festival because , in a year plagued by brothers mcmullen-style , earnestly shallow gen-x angst pictures , it seemed to...",positive,0.105,0.730,0.165,0.9968,positive
3,"in the year 2029 , captain leo davidson ( mark wahlberg , "" boogie nights "" ) is training 'his' chimp pericles to pilot a pod from the usaf oberon space station . \nwhen an ele...",negative,0.049,0.868,0.083,0.9398,positive
4,"[note that followups are directed to rec . arts . movies . current-films and rec . arts . movies . startrek . current \nonly , not to rec . arts . sf . movies . \n-moderator] \...",positive,0.031,0.852,0.117,0.9951,positive


### Interpretation questions

- What does the `compound` score represent?
- Find one review that the lexicon method labeled correctly and explain why it likely worked.
- Find one review it labeled incorrectly. What language may have confused it?
- Which kinds of sentiment are hard for a lexicon-based method?

**Expected answer:** `compound` compresses the positive/negative/neutral scores into a single value from -1 (highly negative) to +1 (highly positive). Reviews with strong opinion words and clear context are easiest. Errors often come from sarcasm, negation, or mixed emotion. Lexicon-based models struggle when sentiment depends on context, domain-specific wording, or irony.

> The model is interpretable because we can inspect the underlying scores rather than treating the label as a black box.

In [4]:
vader_accuracy = accuracy_score(vader_df['label'], vader_df['vader_label'])
print(f'VADER accuracy: {vader_accuracy:.3f}')
print(classification_report(vader_df['label'], vader_df['vader_label'], digits=3))
print(confusion_matrix(vader_df['label'], vader_df['vader_label'], labels=['negative', 'positive']))

wrong_examples = vader_df[vader_df['label'] != vader_df['vader_label']].head(5)
wrong_examples[['label', 'vader_label', 'compound', 'review']]


VADER accuracy: 0.635
              precision    recall  f1-score   support

    negative      0.718     0.444     0.549      1000
    positive      0.598     0.826     0.694      1000

    accuracy                          0.635      2000
   macro avg      0.658     0.635     0.621      2000
weighted avg      0.658     0.635     0.621      2000

[[444 556]
 [174 826]]


,label,vader_label,compound,review
0,positive,negative,-0.5966,"the verdict : spine-chilling drama from horror maestro stephen king , featuring an outstanding , oscar-winning performance from kathy bates . \ngeez , french and saunders had a..."
3,negative,positive,0.9398,"in the year 2029 , captain leo davidson ( mark wahlberg , "" boogie nights "" ) is training 'his' chimp pericles to pilot a pod from the usaf oberon space station . \nwhen an ele..."
7,positive,negative,-0.9977,"( warning to those who have not seen seven : this review contains some spoiler information . ) \nseven is a powerful , fully-realized film on the level of the silence of the l..."
8,negative,positive,0.9418,"movie reviewers have an obligation to see the good , the bad , and the despicable . \ni originally wrote this review for my college newspaper back in '95 , but i wanted to re-w..."
12,positive,negative,-0.7343,"plot : a human space astronaut accidentally falls upon a planet ruled by apes . \nhe is taken prisoner along with some other humans , and tries his best to escape his simian ca..."


## Part 3: Supervised sentiment classification

Now we compare the lexicon method with a simple supervised baseline using TF-IDF features and logistic regression. This keeps the task grounded in the same IMDb review data without redoing all of the earlier preprocessing work from scratch.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['label'],
    test_size=0.25,
    random_state=42,
    stratify=df['label'],
)

X_train.shape, X_test.shape


((1500,), (500,))

In [6]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_features=5000)),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

train_predictions = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print(f'Train accuracy: {train_accuracy:.3f}')

predictions = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.3f}')
print(classification_report(y_test, predictions, digits=3))
print(confusion_matrix(y_test, predictions, labels=['negative', 'positive']))


Train accuracy: 0.959
Test accuracy: 0.820
              precision    recall  f1-score   support

    negative      0.808     0.840     0.824       250
    positive      0.833     0.800     0.816       250

    accuracy                          0.820       500
   macro avg      0.821     0.820     0.820       500
weighted avg      0.821     0.820     0.820       500

[[210  40]
 [ 50 200]]


### Inspect a few errors

Look at the reviews the supervised model predicted incorrectly. Why do these reviews seem harder than the others?

**Expected answer:** errors often occur when the review is mixed, contains irony or indirect phrasing, or has language that is common in one sentiment but used in a different way. The classifier captures patterns from training data, but it still fails when the signal is subtle or context-dependent.

In [7]:
test_df = pd.DataFrame({'review': X_test, 'label': y_test}).reset_index(drop=True)
test_df['model_prediction'] = predictions
errors = test_df[test_df['label'] != test_df['model_prediction']].head(5)
errors[['label', 'model_prediction', 'review']]


,label,model_prediction,review
2,positive,negative,"there's something about ben stiller that makes him a popular choice among casting directors these days . \nstiller currently has three projects in circulation , and what other ..."
6,negative,positive,"when a pair of films from the same director gets released just three weeks apart , it could mean one of two things : that the recently overworked individual is due for a well-d..."
9,negative,positive,"the yet-to-be-released krippendorf's tribe is being marketed as a family comedy , but buyer beware . \nthis movie can't make up its mind . \nis it a family comedy with vulgar r..."
14,positive,negative,"first , i am not a big fan of the x-files tv series . \ni have nothing against it particularly , i just don't happen to watch it . \nhaving said that , i can now say that i lik..."
15,positive,negative,"by phil curtolo mel gibson ( braveheart ) gave a gripping performance as the father of a young kidnapped boy in ron howard's ransom . \ngibson plays tom mullen , a wealthy busi..."


## Part 4: Compare the two approaches

Run both models on the same small set of selected reviews and compare their outputs directly.

In [8]:
selected = test_df.sample(6, random_state=42).copy()
selected['vader_prediction'] = selected['review'].apply(lambda r: analyzer.polarity_scores(r)['compound'])
selected['vader_prediction'] = selected['vader_prediction'].apply(
    lambda score: 'positive' if score >= 0.05 else 'negative' if score <= -0.05 else ('positive' if score >= 0 else 'negative')
)
selected['model_prediction'] = model.predict(selected['review'])
selected[['review', 'label', 'vader_prediction', 'model_prediction']].reset_index(drop=True)


,review,label,vader_prediction,model_prediction
0,"the heist of a flawless , 84 carat diamond , fixed bare knuckle fights , irish gypsies and a dog are just some of the goings on in director guy ritchies's sophomore feature , ""...",positive,positive,positive
1,one never quite knows what one is going to get with a mamet film . \nhis american buffalo is a set-bound piece that has very little plot . \nsometimes he will tell a story that...,positive,positive,negative
2,"aspiring broadway composer robert ( aaron williams ) secretly carries a torch for his best friend , struggling actor marc ( michael shawn lucas ) . \nthe problem is , marc only...",negative,negative,positive
3,"originally launched in 1978 , this popular film was re-introduced in 1998 to a whole new generation of moviegoers . \nbased on the mighty successful musical from broadway , gre...",positive,positive,positive
4,the lives of older people in the twilight of their years attempting to come to grips with their shared histories and possible futures is a fascinating topic . \nfinding an all-...,negative,positive,negative
5,"the marvelous british actor derek jacobi stars in writer and director john maybury's love is the devil about the popular modern artist francis bacon . \nas always , jacobi's ac...",negative,positive,negative


### Compare the predictions

- Where do both methods agree?
- Where do they disagree?
- Which approach seems more reliable on this IMDb sample?
- In what situations would a lexicon-based method still be useful?

**Expected answer:** both models agree on the most strongly sentiment-bearing reviews. They disagree on borderline or mixed cases, where the lexicon may be thrown off by wording or context. The supervised model is usually more reliable on IMDb because it learns patterns from many examples. A lexicon-based method is still useful for quick, interpretable, low-overhead sentiment scoring in real time or in settings with limited labeled data.

## Part 5: Challenge examples

Before running the next examples, predict whether each one is likely to be easy or difficult for VADER and for the supervised model.

In [9]:
challenge_reviews = [
    'This movie is not terrible, but it is definitely not great either.',
    'I loved the first half, but the ending was so disappointing that I almost left.',
    'The film was unbelievably bad in the most entertaining way.',
    'I did not hate it, but I would not recommend it to a friend.',
]
for review in challenge_reviews:
    scores = analyzer.polarity_scores(review)
    print('REVIEW:', review)
    print(scores)
    print('VADER label:', ('positive' if scores['compound'] >= 0.05 else 'negative' if scores['compound'] <= -0.05 else 'positive' if scores['compound'] >= 0 else 'negative'))
    print('Model label:', model.predict([review])[0])
    print('-' * 80)


REVIEW: This movie is not terrible, but it is definitely not great either.
{'neg': 0.237, 'neu': 0.48, 'pos': 0.284, 'compound': -0.0294}
VADER label: negative
Model label: positive
--------------------------------------------------------------------------------
REVIEW: I loved the first half, but the ending was so disappointing that I almost left.
{'neg': 0.297, 'neu': 0.575, 'pos': 0.128, 'compound': -0.6398}
VADER label: negative
Model label: positive
--------------------------------------------------------------------------------
REVIEW: The film was unbelievably bad in the most entertaining way.
{'neg': 0.253, 'neu': 0.534, 'pos': 0.213, 'compound': -0.1531}
VADER label: negative
Model label: negative
--------------------------------------------------------------------------------
REVIEW: I did not hate it, but I would not recommend it to a friend.
{'neg': 0.157, 'neu': 0.472, 'pos': 0.371, 'compound': 0.5624}
VADER label: positive
Model label: positive
---------------------------

## Wrap-up

A good sentiment model is not just about getting a high overall accuracy score. The goal is to understand why sentiment is tricky: negation, sarcasm, mixed emotion, and context all matter.

**Key takeaway:** lexicon-based methods are fast and interpretable, while supervised models generally capture more nuance on a task like IMDb sentiment classification. Yet both can be fooled by context and sentiment that is not expressed in a straightforward way.